# Full Feature Engineering — All Columns in `data.xlsx`

This notebook rebuilds **every** engineered column currently present in `data.xlsx`, applied to the **full dataset** (all rows, no train/test split) — matching the file as-is.

**Original raw columns:** `URL, Type, Purpose, Area, Bedroom, Bath, Added, Price, Location, Location_city, Source, Price_pkr, Area_unit, Area_sqft, Suspected_outlier`

**Engineered columns added:**
| Column | What it is |
|---|---|
| `Log_price` | log(Price_pkr) — the target, log-transformed |
| `Price_per_sqft` | Price_pkr / Area_sqft |
| `Log_area_sqft` | log(Area_sqft) — linearizes the size-price relationship |
| `Location_median_ppsf` | median Price_per_sqft for that Location (city/global fallback) |
| `Area_per_bedroom` | Area_sqft / Bedroom (space per room) |
| `Bath_bedroom_ratio` | Bath / Bedroom (build-quality signal) |
| `Type_target_enc` | property Type, encoded as mean Log_price per type |
| `City_target_enc` | Location_city, encoded as mean Log_price per city |

⚠️ **Note on this version:** because it's built on the full dataset (not a train/test split), `Location_median_ppsf`, `Type_target_enc`, and `City_target_enc` are fit using *all* rows — including each row's own price. This is fine for exploration/reporting, but if you're training a model, use the leak-safe train/test version instead (fit those 3 columns on train only).


In [1]:
import numpy as np
import pandas as pd

INPUT_PATH = "/mnt/user-data/uploads/combined_sale_only.csv"   # original raw file
OUTPUT_PATH = "/mnt/user-data/outputs/data_full_features.xlsx"
MIN_SAMPLES = 3   # minimum listings in a location before trusting its median price/sqft

df = pd.read_csv(INPUT_PATH)
print(df.shape)
df.head()


(3108, 15)


,url,type,purpose,area,bedroom,bath,added,price,location,location_city,source,price_pkr,area_unit,area_sqft,suspected_outlier
0,https://www.zameen.com/Property/askari_askari_...,Apartment,For Sale,10 Marla,3.0,3.0,33 minutes ago,PKR 3 Crore,"Askari 11, Askari",Lahore,Zameen,30000000.0,Marla,2722.50,False
1,https://www.zameen.com/Property/gulberg_3_gulb...,Other,For Sale,2.4 Kanal,6.0,7.0,1 hour ago,PKR 15.5 Crore,"Gulberg 3 - Block M, Gulberg 3",Lahore,Zameen,155000000.0,Kanal,13068.00,False
2,https://www.zameen.com/Property/dha_phase_5_pe...,Apartment,For Sale,9.8 Marla,3.0,4.0,3 hours ago,PKR 6.95 Crore,"Penta Square By DHA Lahore, DHA Phase 5",Lahore,Zameen,69500000.0,Marla,2668.05,False
3,https://www.zameen.com/Property/dha_phase_7_dh...,House,For Sale,1 Kanal,5.0,7.0,3 hours ago,PKR 14.5 Crore,"DHA Phase 7 - Block Y, DHA Phase 7",Lahore,Zameen,145000000.0,Kanal,5445.00,False
4,https://www.zameen.com/Property/dha_phase_7_dh...,House,For Sale,1 Kanal,5.0,6.0,3 hours ago,PKR 14.5 Crore,"DHA Phase 7 - Block U, DHA Phase 7",Lahore,Zameen,145000000.0,Kanal,5445.00,False


## 1. Basic cleanup

Drop rows with non-positive price or area (would break the log transform).

In [2]:
before = len(df)
df = df[(df['price_pkr'] > 0) & (df['area_sqft'] > 0)].copy()
print(f"{before} -> {len(df)} rows after removing non-positive price/area")


3108 -> 3108 rows after removing non-positive price/area


## 2. `Log_price` and `Price_per_sqft`

Base calculations everything else builds on.

In [3]:
df['log_price'] = np.log(df['price_pkr'])
df['price_per_sqft'] = df['price_pkr'] / df['area_sqft']
df[['price_pkr', 'log_price', 'area_sqft', 'price_per_sqft']].describe()


,price_pkr,log_price,area_sqft,price_per_sqft
count,3.108000e+03,3108.000000,3108.000000,3108.000000
mean,6.973082e+07,17.392381,3401.614664,20390.775820
std,2.233383e+08,1.065551,6851.365251,21059.010841
min,4.000000e+05,12.899220,8.000000,133.333333
25%,1.850000e+07,16.733281,1361.250000,11684.137717
50%,3.500000e+07,17.370859,2160.000000,16078.971534
75%,7.000000e+07,18.064006,4500.000000,21854.912764
max,9.000000e+09,22.920490,217800.000000,525000.000000


## 3. `Log_area_sqft`

Log-transformed area — price grows fast for small properties and slows down for huge ones, so logging area straightens that curve.

In [4]:
df['log_area_sqft'] = np.log(df['area_sqft'])


## 4. `Location_median_ppsf`

Median price-per-sqft for each `location`. Falls back to `location_city` median, then the global median, when a location has fewer than `MIN_SAMPLES` listings.

In [5]:
loc_counts = df.groupby('location')['price_per_sqft'].count()
loc_median = df.groupby('location')['price_per_sqft'].median()
reliable_locations = loc_counts[loc_counts >= MIN_SAMPLES].index
loc_median_reliable = loc_median.loc[reliable_locations]
city_median_ppsf = df.groupby('location_city')['price_per_sqft'].median()
global_median_ppsf = df['price_per_sqft'].median()

def map_location_median(row):
    if row['location'] in loc_median_reliable.index:
        return loc_median_reliable.loc[row['location']]
    elif row['location_city'] in city_median_ppsf.index:
        return city_median_ppsf.loc[row['location_city']]
    return global_median_ppsf

df['location_median_ppsf'] = df.apply(map_location_median, axis=1)
df[['location', 'location_median_ppsf']].drop_duplicates().head()


,location,location_median_ppsf
0,"Askari 11, Askari",16332.152696
1,"Gulberg 3 - Block M, Gulberg 3",15794.306703
2,"Penta Square By DHA Lahore, DHA Phase 5",15794.306703
3,"DHA Phase 7 - Block Y, DHA Phase 7",20202.020202
4,"DHA Phase 7 - Block U, DHA Phase 7",17860.422406


## 5. `Area_per_bedroom`

`area_sqft / bedroom`. About 10% of listings (mostly Plots) have `bedroom = 0` — for those we use the full `area_sqft` instead of dividing by zero.

In [6]:
df['area_per_bedroom'] = np.where(df['bedroom'] > 0, df['area_sqft'] / df['bedroom'], df['area_sqft'])


## 6. `Bath_bedroom_ratio`

`bath / bedroom`, a build-quality/luxury signal. Set to 0 where `bedroom = 0`.

In [7]:
df['bath_bedroom_ratio'] = np.where(df['bedroom'] > 0, df['bath'] / df['bedroom'], 0)


## 7. `Type_target_enc`

Property type, target-encoded as the mean `log_price` for that type.

In [8]:
type_target_map = df.groupby('type')['log_price'].mean()
df['type_target_enc'] = df['type'].map(type_target_map)
df[['type', 'type_target_enc']].drop_duplicates().sort_values('type_target_enc', ascending=False)


,type,type_target_enc
547,Building,19.045737
2244,Commercial Plot,18.268259
1,Other,17.994220
1395,Penthouse,17.865724
2361,Factory,17.822844
51,Farm House,17.550735
3,House,17.538551
1655,Office,17.483480
432,Upper Portion,17.314357
7,Room,17.284656


## 8. `City_target_enc`

Same target-encoding approach, applied to `location_city`.

In [9]:
city_target_map = df.groupby('location_city')['log_price'].mean()
df['city_target_enc'] = df['location_city'].map(city_target_map)
df[['location_city', 'city_target_enc']].drop_duplicates().sort_values('city_target_enc', ascending=False)


,location_city,city_target_enc
748,Islamabad,17.646460
0,Lahore,17.480455
374,Karachi,17.387238
1494,Faisalabad,17.295914
1121,Rawalpindi,17.207726
1869,Multan,17.118243


## 9. Final check — all columns present, no missing values

In [10]:
print(df.shape)
print(df.columns.tolist())
print()
print("Missing values per column:")
print(df.isnull().sum())


(3108, 23)
['url', 'type', 'purpose', 'area', 'bedroom', 'bath', 'added', 'price', 'location', 'location_city', 'source', 'price_pkr', 'area_unit', 'area_sqft', 'suspected_outlier', 'log_price', 'price_per_sqft', 'log_area_sqft', 'location_median_ppsf', 'area_per_bedroom', 'bath_bedroom_ratio', 'type_target_enc', 'city_target_enc']

Missing values per column:
url                     0
type                    0
purpose                 0
area                    0
bedroom                 0
bath                    0
added                   0
price                   0
location                0
location_city           0
source                  0
price_pkr               0
area_unit               0
area_sqft               0
suspected_outlier       0
log_price               0
price_per_sqft          0
log_area_sqft           0
location_median_ppsf    0
area_per_bedroom        0
bath_bedroom_ratio      0
type_target_enc         0
city_target_enc         0
dtype: int64


## 10. Save to Excel

In [11]:
# Match the column naming style of data.xlsx (Title_Case)
df_out = df.rename(columns={
    'url': 'URL', 'type': 'Type', 'purpose': 'Purpose', 'area': 'Area',
    'bedroom': 'Bedroom', 'bath': 'Bath', 'added': 'Added', 'price': 'Price',
    'location': 'Location', 'location_city': 'Location_city', 'source': 'Source',
    'price_pkr': 'Price_pkr', 'area_unit': 'Area_unit', 'area_sqft': 'Area_sqft',
    'suspected_outlier': 'Suspected_outlier', 'log_price': 'Log_price',
    'price_per_sqft': 'Price_per_sqft', 'log_area_sqft': 'Log_area_sqft',
    'location_median_ppsf': 'Location_median_ppsf', 'area_per_bedroom': 'Area_per_bedroom',
    'bath_bedroom_ratio': 'Bath_bedroom_ratio', 'type_target_enc': 'Type_target_enc',
    'city_target_enc': 'City_target_enc',
})

df_out.to_excel(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}  |  shape: {df_out.shape}")


Saved: /mnt/user-data/outputs/data_full_features.xlsx  |  shape: (3108, 23)
